# Prompt creation functions

In [ ]:
pip install openai tiktoken

## Missing value imputation prompt

In [ ]:
def create_prompt(
    year=None,
    manufacturer=None,
    model=None,
    description=None,
    condition=None,
    cylinders=None,
    drive=None,
    vehicle_type=None,
):

    return f"""
You extract structured vehicle information.

Rules
- Keep existing values unchanged.
- Infer ONLY missing values.
- Use all available information consistently.
- If a field cannot be inferred confidently, return null.
- When a field specifies allowed values, return ONLY one of those values.
- Do not invent categories or values.
- Return ONLY valid JSON.

Vehicle

year: {year}
manufacturer: {manufacturer}
model: {model}
condition: {condition}
cylinders: {cylinders}
drive: {drive}
type: {vehicle_type}

Description

{description}

====================
FIELD RULES
====================

MANUFACTURER RULES
Return the vehicle manufacturer.

MODEL RULES
Return the canonical base model only.
Do NOT merge different model families

Remove:
- trim levels
- editions
- packages
- drivetrain
- engine names
- transmission
- body style
- cab configuration
- colours
- years

Examples:
F-150 XLT SuperCrew -> F-150
Silverado 2500HD LT -> Silverado 2500HD
Camry XLE Hybrid -> Camry
Accord Sport 2.0T -> Accord

CONDITION RULES
Return ONLY one of:

new
like new
excellent
good
fair
salvage

CYLINDERS RULES
Return ONLY one of:

3 cylinders
4 cylinders
5 cylinders
6 cylinders
8 cylinders
10 cylinders
12 cylinders
other

DRIVE RULES
Return ONLY one of:

fwd
rwd
4wd

TYPE RULES
Return ONLY one of:

pickup
truck
suv
sedan
coupe
hatchback
mini-van
convertible
wagon
van
offroad
bus
other

Type means the VEHICLE BODY STYLE.

Do NOT return:
- fuel types
- engine names
- trim levels
- drivetrain
- commercial descriptions

Return the closest matching body style.
====================
RETURN JSON
====================

{{
    "manufacturer": null,
    "model": null,
    "condition": null,
    "cylinders": null,
    "drive": null,
    "type": null
}}
""".strip()

## Normalization prompts

### Manufacturer normalization

In [ ]:
def create_manufacturer_normalization_prompt(manufacturers):
    return f"""
You are an automotive data normalization assistant.

Normalize every manufacturer to ONE canonical manufacturer.

Rules:

- Every input value MUST appear exactly once.
- Every input value MUST be a key in the output.
- If already canonical, return it unchanged.
- Ignore capitalization and whitespace.
- Correct common misspellings.
- Do NOT invent manufacturers.
- Every output MUST be one of the canonical manufacturers.
- If uncertain, return "other".
- Return ONLY valid JSON.

Canonical manufacturers:

gmc, chevrolet, toyota, ford, jeep, nissan, ram, mazda, cadillac,
honda, dodge, lexus, jaguar, buick, chrysler, volvo, audi,
infiniti, lincoln, alfa-romeo, subaru, acura, hyundai,
mercedes-benz, bmw, mitsubishi, volkswagen, porsche,
kia, land rover, ferrari, mini, pontiac, fiat,
tesla, saturn, mercury, harley-davidson,
datsun, aston-martin, morgan, other

Manufacturers:

{manufacturers}

Return ONLY a JSON dictionary.

Example:

{{
    "vw":"volkswagen",
    "chevy":"chevrolet",
    "merc":"mercedes-benz"
}}
""".strip()


### Model prompts (For all known manufacturers and others)

In [ ]:
def create_model_normalization_prompt(manufacturer, models):

    return f"""
You are an automotive model normalization assistant.

Manufacturer:
{manufacturer}

Every model below belongs to this manufacturer.

RULES

- Every input model must appear exactly once.
- Return only valid JSON.
- Never change the manufacturer.
- If already canonical, return it unchanged.
- If the canonical base model cannot be determined confidently, return null.
- Never invent model names.

Normalize by removing trim, edition, package, drivetrain, engine, transmission, body style, cab configuration, marketing words, colours and years.

Preserve model families and heavy-duty variants.

If the input is only a trim, engine, package, colour or drivetrain, return null.

Never merge different model families.

Examples

F-150 XLT SuperCrew 4x4 -> F-150
Silverado 2500HD Duramax -> Silverado 2500HD
Camry LE Hybrid -> Camry
Wrangler Rubicon Unlimited -> Wrangler Unlimited

Do NOT merge

F-150 != F-250
Bronco != Bronco Sport
Silverado 1500 != Silverado 2500HD
Corolla != Corolla Cross

Return null

Lariat -> null
Duramax -> null
Crew Cab -> null
Camry Corolla -> null

MODELS

{models}

Return JSON

{{
    "f150 xlt": "F-150",
    "silverado 2500hd duramax": "Silverado 2500HD",
    "crew cab": null
}}
""".strip()

In [ ]:
def create_other_model_normalization_prompt(models):

    return f"""
You are an automotive manufacturer and model normalization assistant.

The manufacturer is unknown.

Canonical manufacturers:

gmc, chevrolet, toyota, ford, jeep, nissan, ram, mazda,
cadillac, honda, dodge, lexus, jaguar, buick, chrysler,
volvo, audi, infiniti, lincoln, alfa-romeo, subaru,
acura, hyundai, mercedes-benz, bmw, mitsubishi,
volkswagen, porsche, kia, land rover, ferrari, mini,
pontiac, fiat, tesla, saturn, mercury,
harley-davidson, datsun, aston-martin, morgan, other

RULES

- Every input model must appear exactly once.
- Return only valid JSON.
- Never invent manufacturers or models.
- Infer the manufacturer only if the model uniquely identifies one.
- Otherwise return null for both manufacturer and model.

Normalize by removing trim, edition, package, drivetrain, engine, transmission, body style, cab configuration, marketing words, colours and years.

Preserve model families and heavy-duty variants.

If the input is only a trim, engine, package or drivetrain, return null for both fields.

Examples

Camry XLE
-> manufacturer: toyota
-> model: Camry

Silverado 1500 LT
-> manufacturer: chevrolet
-> model: Silverado 1500

F150 XLT
-> manufacturer: ford
-> model: F-150

Duramax
-> manufacturer: null
-> model: null

Crew Cab
-> manufacturer: null
-> model: null

Camry Corolla
-> manufacturer: null
-> model: null

MODELS

{models}

Return JSON

{{
    "camry xle": {{
        "manufacturer": "toyota",
        "model": "Camry"
    }},
    "duramax": {{
        "manufacturer": null,
        "model": null
    }}
}}
""".strip()

# Load data (POST-PHASE 2)

## Load CSV

In [ ]:
import pandas as pd

processed_df = pd.read_parquet("/content/drive/MyDrive/processed_df.parquet")
print(processed_df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 389309 entries, 0 to 389308
Data columns (total 23 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   Unnamed: 0    389309 non-null  int64  
 1   id            389309 non-null  int64  
 2   region        389309 non-null  object 
 3   price         389309 non-null  int64  
 4   year          389309 non-null  float64
 5   manufacturer  374376 non-null  object 
 6   model         384708 non-null  object 
 7   condition     241645 non-null  object 
 8   cylinders     229443 non-null  object 
 9   fuel          387006 non-null  object 
 10  odometer      387082 non-null  float64
 11  title_status  381879 non-null  object 
 12  transmission  387548 non-null  object 
 13  drive         269890 non-null  object 
 14  type          303843 non-null  object 
 15  paint_color   272602 non-null  object 
 16  image_url     389309 non-null  object 
 17  description   389307 non-null  object 
 18  stat

In [ ]:
processed_df['id'].is_unique

True

## Create new DF containing normalization + imputation targets

In [ ]:
TARGETS = [
    "manufacturer",
    "model",
    "condition",
    "cylinders",
    "drive",
    "type",
]

llm_df = (
    processed_df[
        processed_df[TARGETS].isna().any(axis=1)
    ]
    .copy()
)

print(f"Rows requiring LLM: {len(llm_df):,}")

ids = llm_df["id"].tolist()

Rows requiring LLM: 264,594


# Imputation

## Prepare imputation prompt script

In [ ]:
MAX_DESCRIPTION_CHARS = 800

def prepare_llm_input(row):

    description = row["description"]

    if pd.isna(description):
        description = None
    else:
        description = str(description).strip()[:MAX_DESCRIPTION_CHARS]

    return {
        "id": row["id"],
        "year": row["year"],
        "manufacturer": row["manufacturer"],
        "model": row["model"],
        "condition": row["condition"],
        "cylinders": row["cylinders"],
        "drive": row["drive"],
        "vehicle_type": row["type"],
        "description": description,
    }

## Model

In [ ]:
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    pipeline,
)

MODEL_NAME = "Qwen/Qwen3-4B-Instruct-2507"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
)

model.eval()

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
)

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.38k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

...

model.safetensors.index.json:   0%|          | 0.00/32.8k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

## Run model for imputation

In [ ]:
prompts = []

for _, row in llm_df.iterrows():
    llm_input = prepare_llm_input(row)
    prompt = create_prompt(
        year=llm_input["year"],
        manufacturer=llm_input["manufacturer"],
        model=llm_input["model"],
        condition=llm_input["condition"],
        cylinders=llm_input["cylinders"],
        drive=llm_input["drive"],
        vehicle_type=llm_input["vehicle_type"],
        description=llm_input["description"],
    )
    messages = [
        {
            "role": "system",
            "content": "You are an automotive information extraction assistant."
        },
        {
            "role": "user",
            "content": prompt
        }
    ]
    prompts.append(
        tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )
    )

In [ ]:
import os
import json
from tqdm.auto import tqdm
import torch

# Uncomment for a quick sanity test.
save_dir = "/content/drive/MyDrive/used_car"

# Create the folder if it doesn't exist
os.makedirs(save_dir, exist_ok=True)

BATCH_SIZE = 128
MAX_NEW_TOKENS = 120

predictions = []

print(f"Rows: {len(prompts):,}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Expected batches: {(len(prompts)+BATCH_SIZE-1)//BATCH_SIZE}")

for batch_idx, start in enumerate(
    tqdm(range(0, len(prompts), BATCH_SIZE))
):

    batch = prompts[start:start + BATCH_SIZE]

    outputs = pipe(
        batch,
        batch_size=BATCH_SIZE,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=False,
        return_full_text=False,
    )

    for i, output in enumerate(outputs):
        text = output[0]["generated_text"].strip()
        try:
            pred = json.loads(text)

        except Exception:
            pred = {
                "error": text
            }
        # Attach the correct id here
        pred["id"] = ids[start + i]
        assert pred["id"] == ids[start + i]

        predictions.append(pred)

    print(
        f"Batch {batch_idx + 1}, GPU Allocated: {torch.cuda.memory_allocated()/1024**3:.2f} GB, GPU Reserved: {torch.cuda.memory_reserved()/1024**3:.2f} GB"
    )

    if (batch_idx + 1) % 50 == 0:
      with open(
          f"{save_dir}/llm_impute_predictions_checkpoint.json",
          "w",
      ) as f:
          json.dump(
              predictions,
              f,
              indent=4,
              ensure_ascii=False,
          )

with open(
    f"{save_dir}/llm_impute_predictions.json",
    "w",
) as f:

    json.dump(
        predictions,
        f,
        indent=4,
        ensure_ascii=False,
    )

Rows: 264,594
Batch size: 128
Expected batches: 2068


  0%|          | 0/2068 [00:00<?, ?it/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=120) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Batch 1, GPU Allocated: 7.50 GB, GPU Reserved: 34.98 GB


[transformers] Both `max_new_tokens` (=120) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

Batch 2065, GPU Allocated: 7.50 GB, GPU Reserved: 66.52 GB


[transformers] Both `max_new_tokens` (=120) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Batch 2066, GPU Allocated: 7.50 GB, GPU Reserved: 66.52 GB


[transformers] Both `max_new_tokens` (=120) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=120) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Batch 2067, GPU Allocated: 7.50 GB, GPU Reserved: 66.52 GB
Batch 2068, GPU Allocated: 7.50 GB, GPU Reserved: 66.52 GB


# Post-imputation analysis and merging

In [ ]:
import json
import pandas as pd

with open("/content/drive/MyDrive/used_car/llm_impute_predictions.json", "r", encoding="utf-8") as f:
    predictions = json.load(f)

pred_df = pd.DataFrame(predictions)

# print(pred_df.shape)
# print(pred_df["id"].is_unique)


# for item in TARGETS:
#   print(pred_df[item].nunique())
#   print(pred_df[item].unique())

print(len(ids))
print(len(predictions))
print(len(pred_df))
print(pred_df["id"].is_unique)
print(pred_df["id"].isna().sum())
print((pred_df["id"].to_numpy() == ids).all())
for i in [0, 1, 10, 100, 1000, len(ids)-1]:
    print(
        ids[i],
        pred_df.iloc[i]["id"]
    )
print(set(ids) - set(pred_df["id"]))
print(set(pred_df["id"]) - set(ids))


264594
264594
264594
True
0
True
7316814884 7316814884
7316814758 7316814758
7314910156 7314910156
7316788498 7316788498
7303800718 7303800718
7301591129 7301591129
set()
set()


In [ ]:
print(TARGETS)
merged_df = processed_df.merge(
    pred_df[
        ["id"] + TARGETS
    ],
    on="id",
    how="left",
    suffixes=("", "_llm"),
)

['manufacturer', 'model', 'condition', 'cylinders', 'drive', 'type']


In [ ]:
for col in TARGETS:

    mask = (
        merged_df[col].isna()
        & merged_df[f"{col}_llm"].notna()
    )

    merged_df.loc[mask, col] = merged_df.loc[
        mask,
        f"{col}_llm"
    ]

for col in TARGETS:
  merged_df[col] = merged_df[col].apply(lambda x: str(x).lower().strip())

In [ ]:
for item in TARGETS:
  print(item)
  print(merged_df[item].nunique())

manufacturer
463
model
27302
condition
38
cylinders
16
drive
15
type
32


# Normalization

## Manufacturer normalization

### Open AI loading

In [ ]:
from openai import OpenAI
from google.colab import userdata
import json
import os

client = OpenAI(
    api_key=userdata.get('OPENAI_API_KEY')
)

In [ ]:
import json
from openai import OpenAI

def run_prompt(prompt, model="gpt-5-mini"):
    response = client.responses.create(
        model=model,
        input=prompt,
        text={
            "format": {
                "type": "json_object"
            }
        }
    )

    return json.loads(response.output_text)

### Manufacturers normalization

In [ ]:
manufacturers = (
    merged_df["manufacturer"]
    .dropna()
    .astype(str)
    .str.strip()
    .str.lower()
    .unique()
    .tolist()
)

In [ ]:
manufacturer_prompt = create_manufacturer_normalization_prompt(
    manufacturers
)

manufacturer_mapping = run_prompt(manufacturer_prompt, model = "gpt-5.6-luna")

In [ ]:
import os
import json
from tqdm.auto import tqdm
import torch

# Uncomment for a quick sanity test.
save_dir = "/content/drive/MyDrive/used_car"

# Create the folder if it doesn't exist
os.makedirs(save_dir, exist_ok=True)

with open(
    f"{save_dir}/manufacturer_mapping.json",
    "w",
) as f:
    json.dump(
        manufacturer_mapping,
        f,
        indent=4,
        ensure_ascii=False,
    )

In [ ]:
import pandas as pd

merged_df["manufacturer"] = (
    merged_df["manufacturer"]
    .astype(str)
    .str.strip()
    .str.lower()
    .replace(manufacturer_mapping)
)
merged_df["manufacturer"] = (
    merged_df["manufacturer"]
    .replace(["nan", "none", "null"], pd.NA)
)

print(merged_df["manufacturer"].nunique())
print(merged_df["manufacturer"].value_counts(dropna=False))

43
manufacturer
ford               65084
chevrolet          51015
toyota             31686
honda              21209
jeep               17485
nissan             17368
ram                16415
gmc                15498
bmw                13320
dodge              12457
mercedes-benz      10296
other               9594
hyundai             9461
volkswagen          9061
subaru              9037
lexus               7553
kia                 7488
audi                7122
cadillac            6616
chrysler            5834
acura               5712
buick               5189
mazda               5136
infiniti            4489
lincoln             4016
volvo               3298
mitsubishi          3139
pontiac             2270
mini                2239
land rover          1997
jaguar              1915
porsche             1291
mercury             1152
saturn              1074
tesla                852
alfa-romeo           849
fiat                 777
harley-davidson      140
ferrari               84
datsun   

## Model normalization

In [ ]:
for u_val in merged_df["manufacturer"].unique():
  print(f"Manufacturer {u_val} has {merged_df[merged_df["manufacturer"] == u_val]["model"].nunique()} unique models")

Manufacturer gmc has 1256 unique models
Manufacturer chevrolet has 3072 unique models
Manufacturer toyota has 1768 unique models
Manufacturer ford has 4105 unique models
Manufacturer jeep has 781 unique models
Manufacturer nissan has 1028 unique models
Manufacturer ram has 1011 unique models
Manufacturer mazda has 439 unique models
Manufacturer cadillac has 473 unique models
Manufacturer honda has 1089 unique models
Manufacturer dodge has 909 unique models
Manufacturer lexus has 464 unique models
Manufacturer jaguar has 212 unique models
Manufacturer buick has 341 unique models
Manufacturer chrysler has 442 unique models
Manufacturer volvo has 393 unique models
Manufacturer audi has 627 unique models
Manufacturer infiniti has 352 unique models
Manufacturer lincoln has 257 unique models
Manufacturer alfa-romeo has 42 unique models
Manufacturer subaru has 637 unique models
Manufacturer acura has 317 unique models
Manufacturer hyundai has 537 unique models
Manufacturer mercedes-benz has 1

In [ ]:
from tqdm.auto import tqdm
import time

def to_hour_minute_second(seconds):
    hours, remainder = divmod(seconds, 3600)
    minutes, seconds = divmod(remainder, 60)

    return f"{int(hours):02}:{int(minutes):02}:{int(seconds):02}"

BATCH_SIZE = 100

model_mapping = {}

manufacturers = (
    merged_df["manufacturer"]
    .dropna()
    .unique()
)

overall_start = time.perf_counter()

for manufacturer in manufacturers:

    manufacturer_start = time.perf_counter()

    unique_models = (
        merged_df.loc[
            merged_df["manufacturer"] == manufacturer,
            "model"
        ]
        .dropna()
        .astype(str)
        .str.strip()
        .str.lower()
        .unique()
        .tolist()
    )

    print(f"{manufacturer}: {len(unique_models)} models")

    for start in tqdm(range(0, len(unique_models), BATCH_SIZE), leave=False):

        batch = unique_models[start:start + BATCH_SIZE]

        if manufacturer == "other":

            prompt = create_other_model_normalization_prompt(
                batch
            )

        else:

            prompt = create_model_normalization_prompt(
                manufacturer,
                batch
            )

        result = run_prompt(prompt, model = "gpt-5.6-luna")
        model_mapping.update(result)

    manufacturer_time  = time.perf_counter() - manufacturer_start
    print(f"{manufacturer} took {to_hour_minute_second(manufacturer_time)}")

overall_time = time.perf_counter() - overall_start
print(f"Total time: {to_hour_minute_second(overall_time)}")


with open("/content/drive/MyDrive/used_car/gpt_output_normalization_1.json", "w") as file:
    json.dump(model_mapping, file, indent=4)

gmc: 1256 models


  0%|          | 0/13 [00:00<?, ?it/s]

gmc took 00:04:32
chevrolet: 3072 models


  0%|          | 0/31 [00:00<?, ?it/s]

chevrolet took 00:13:38
toyota: 1768 models


...

datsun took 00:00:07
aston-martin: 11 models


  0%|          | 0/1 [00:00<?, ?it/s]

aston-martin took 00:00:04
morgan: 4 models


  0%|          | 0/1 [00:00<?, ?it/s]

morgan took 00:00:02
Total time: 01:51:35


In [ ]:
import json

with open("/content/drive/MyDrive/used_car/gpt_output_normalization_1.json") as f:
    model_mapping = json.load(f)

print(len(model_mapping))

27245


In [ ]:
clean_model_mapping = {}
manufacturer_inference = {}

for key, value in model_mapping.items():

    if isinstance(value, dict):
        clean_model_mapping[key] = value["model"]
        manufacturer_inference[key] = value["manufacturer"]

    else:
        clean_model_mapping[key] = value


merged_df["model"] = (
    merged_df["model"]
    .astype(str)
    .str.strip()
    .str.lower()
    .replace(clean_model_mapping)
)

for item in TARGETS:
  print(item)
  print(merged_df[item].nunique())

manufacturer
43
model
2104
condition
38
cylinders
16
drive
15
type
32
